# Chapter 5: Logistic Regression


<!-- Macro definitions for MathJax, mirroring book.tex -->
$$
\newcommand{\bm}[1]{\boldsymbol{#1}}
\newcommand{\Det}[1]{|\boldsymbol{#1}|}
\newcommand{\bigO}{\mathcal{O}}
\newcommand{\var}{\mathrm{Var}}
\newcommand{\cov}{\mathrm{Cov}}
\newcommand{\Prob}{\mathrm{Prob}}
\newcommand{\mean}[1]{\langle #1 \rangle}
$$

Everything so far has predicted a continuous number.  We now turn to
*classification*, where the target is a label drawn from a finite set:
whether a tumour is malignant or benign, whether a credit card holder will
default, which of ten digits a handwritten image shows.  The change of target
forces a change of model and, more interestingly, a change of loss function --
and the new loss will turn out to follow from the same maximum-likelihood
argument that produced least squares in Section *Deriving least squares from a probability distribution*.

Logistic regression is the natural first classifier, and it occupies the same
position in this book that linear regression did.  It is simple enough that
everything can be computed, its cost function is convex so that the
optimisation theory of Chapter 4 applies without qualification,
and -- most importantly for what follows -- it is exactly a neural network with
no hidden layers.  The sigmoid we introduce here is an activation function,
the cross-entropy we derive here is the loss used to train classifiers
throughout deep learning, and the gradient we compute here reappears as the
last step of backpropagation.  A reader who understands this chapter
thoroughly has already met most of the ingredients of the next one.


## Classification problems

We consider the case where the responses $y_i$ are discrete and take values
from $k=0,\dots,K-1$, that is $K$ classes.  The goal is to predict the class
from the design matrix $\bm{X}\in\mathbb{R}^{n\times p}$ of
Eq. (1.1), made of $n$ samples each carrying $p$ features,
and in particular to identify the classes of new, unseen samples.

We specialise for most of the chapter to two classes, with outputs $y_i=0$ and
$y_i=1$.  The outcome might represent whether a credit card holder defaults on
their debt,

$$
y_i = \begin{cases} 0 & \text{no},\\ 1 & \text{yes}. \end{cases}\tag{5.1}
$$

**Why not simply use linear regression?.** 
Before introducing anything new, it is worth seeing why the machinery of
Chapter 3 is not adequate.  We could fit the linear model

$$
\bm{y} = \bm{X}\bm{\theta} + \bm{\varepsilon}\tag{5.2}
$$

and classify by thresholding, predicting class $1$ when
$\tilde{y}_i>0.5$ and class $0$ otherwise.  Something like this can be made
to work, but it is unsatisfactory for a fundamental reason: the right-hand
side of Eq. (5.2) takes values on the entire real axis,
while $y_i$ takes only the values $0$ and $1$.  A fitted value of $-3.7$ or
$+2.4$ has no interpretation.  Worse, a single distant point drags the fitted
line and moves the decision threshold, so that adding a strongly-classified
observation can cause a previously correct classification to flip.

One way to force a discrete output is to compose the linear model with a sign
function, $f(s_i)=1$ if $s_i\ge0$ and $0$ otherwise.  This is the
*perceptron*, historically the first machine learning model and the
subject of the opening of the next chapter.  It is a *hard* classifier:
each data point is assigned deterministically to a category.  In many
situations we would rather have a *soft* classifier, one that outputs
the *probability* of a given category.  Probabilities can be thresholded
when a decision is needed, but they also express confidence, they can be
combined with costs when different errors matter differently, and -- as we
shall see -- they lead to a differentiable cost function, which the step
function does not.

**An illustration.** 
Data on coronary heart disease (CHD) as a function of age make the point.
Plotting whether a person has had CHD ($y=1$) or not ($y=0$) against age gives
two horizontal bands of points, and a straight-line fit through them is
visibly meaningless.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

chd = pd.read_csv("DataFiles/chddata.csv", names=("ID", "Age", "Agegroup", "CHD"))
plt.scatter(chd["Age"], chd["CHD"], marker="o")
plt.axis([18, 70.0, -0.1, 1.2])
plt.xlabel("Age"); plt.ylabel("CHD")
plt.show()


What is meaningful is the *mean* value within each age group.


In [ ]:
agegroupmean = np.array([0.1, 0.133, 0.250, 0.333, 0.462, 0.625, 0.765, 0.800])
group = np.array([1, 2, 3, 4, 5, 6, 7, 8])
plt.plot(group, agegroupmean, "r-")
plt.axis([0, 9, 0, 1.0])
plt.xlabel("Age group"); plt.ylabel("CHD mean value")
plt.show()


Two features of the resulting curve are decisive.  It is confined to the
interval $[0,1]$, as any mean of zeros and ones must be.  And it is
*S-shaped*: nearly flat at both ends, steepest in the middle.  We are
looking for a function $f(y_i\mid x_i)$ giving the expected value of the
output for a given input; in linear regression this was
$f(y_i\mid x_i)=\theta_0+\theta_1x_i$, which ranges over the whole real line,
whereas here we need $0\le f(y_i\mid x_i)\le1$ together with the observed
S-shape.  A function with exactly these properties is the subject of the next
section, and we shall interpret it as the probability of observing $y_i=1$ for
a given $x_i$.


## The logistic function

The *logistic* or *sigmoid* function is

$$
\sigma(t) = \frac{1}{1+\exp(-t)} = \frac{\exp(t)}{1+\exp(t)} .\tag{5.3}
$$

It maps the whole real line onto the open interval $(0,1)$, tends to $0$ as
$t\to-\infty$ and to $1$ as $t\to+\infty$, and takes the value $\tfrac12$ at
the origin.  It has the S-shape observed in the CHD group means.

Three properties will be used repeatedly.  The first is the reflection
identity

$$
1-\sigma(t) = \sigma(-t),\tag{5.4}
$$

immediate from Eq. (5.3), which says that the probability of
class $0$ is obtained from the probability of class $1$ by flipping the sign
of the argument.  The second is the derivative,

$$
\boxed{\;
  \frac{d\sigma}{dt} = \sigma(t)\left[1-\sigma(t)\right] . \;}\tag{5.5}
$$

This is worth deriving once.  Writing $\sigma=(1+e^{-t})^{-1}$,

$$
\frac{d\sigma}{dt} = \frac{e^{-t}}{\left(1+e^{-t}\right)^{2}}
   = \frac{1}{1+e^{-t}}\cdot\frac{e^{-t}}{1+e^{-t}}
   = \sigma(t)\left[1-\sigma(t)\right],
$$

using $e^{-t}/(1+e^{-t})=1-\sigma(t)$.  Equation (5.5) is
remarkable in that the derivative is expressed in terms of the function value
alone, with no further exponentials to evaluate.  Every gradient in this
chapter, and every sigmoid backpropagation step in the next, rests on it.
Note also that the derivative is largest at $t=0$, where it equals $1/4$, and
falls off rapidly in both directions -- a fact we shall return to when
discussing why deep networks built from sigmoids are hard to train.

The third property is the *logit*, the inverse function,

$$
t = \ln\frac{\sigma}{1-\sigma},\tag{5.6}
$$

the logarithm of the odds.  Equation (5.6) is what makes logistic
regression *linear*: we shall model the log-odds as a linear function of
the features, even though the probability itself is a non-linear function of
them.

**Related functions.** 
Two relatives appear later.  The *step* function, $f(t)=1$ for $t\ge0$
and $0$ otherwise, is the hard-classifier limit of the sigmoid and is the
perceptron's activation; it is not differentiable, which is precisely why it
cannot be trained by the gradient methods of Chapter 4.  The
hyperbolic tangent

$$
\tanh(t) = \frac{e^{t}-e^{-t}}{e^{t}+e^{-t}} = 2\sigma(2t)-1\tag{5.7}
$$

is a rescaled sigmoid mapping onto $(-1,1)$ rather than $(0,1)$, and is a
common activation function in neural networks because its output is centred on
zero.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.arange(-5, 5, 0.1)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
ax[0].plot(z, 1.0 / (1.0 + np.exp(-z)));      ax[0].set_title("sigmoid")
ax[1].plot(z, np.where(z >= 0.0, 1.0, 0.0));  ax[1].set_title("step")
ax[2].plot(z, np.tanh(z));                    ax[2].set_title("tanh")
for a in ax:
    a.grid(True); a.set_xlabel("z")
plt.show()


## Maximum likelihood and the cross-entropy

We now derive the cost function.  The argument is the one used in
Section *Deriving least squares from a probability distribution* to obtain least squares, with the Gaussian
noise model replaced by a Bernoulli one -- and the notebox there promised
exactly this outcome.

**The model.** 
Assume two classes and, to begin with, a single predictor and two parameters.
We model the probability of class $1$ by a sigmoid applied to a linear
function of the input,

$$
\begin{align}
p(y_i=1\mid x_i,\bm{\theta})
   &= \frac{\exp(\theta_0+\theta_1x_i)}{1+\exp(\theta_0+\theta_1x_i)}
    = \sigma(\theta_0+\theta_1x_i),
  \\
  p(y_i=0\mid x_i,\bm{\theta})
   &= 1 - p(y_i=1\mid x_i,\bm{\theta}),
\end{align}
$$

where $\bm{\theta}=(\theta_0,\theta_1)$ are the weights we wish to determine.
Equation (5.9) is forced: the two probabilities must sum to one.
Taking the logit of Eq. (5.8) using Eq. (5.6) gives the
equivalent statement

$$
\ln\frac{p(y_i=1\mid x_i,\bm{\theta})}{1-p(y_i=1\mid x_i,\bm{\theta})}
   = \theta_0+\theta_1x_i ,\tag{5.10}
$$

which is the cleanest way to say what logistic regression assumes: the
log-odds are linear in the features.

**The likelihood.** 
For a data set $\mathcal{D}=\{(y_i,x_i)\}$ with binary labels and independent
observations, the maximum likelihood principle instructs us to maximise the
probability of the data actually seen.  A single observation contributes
$p(y_i=1\mid x_i,\bm{\theta})$ if $y_i=1$ and
$1-p(y_i=1\mid x_i,\bm{\theta})$ if $y_i=0$; both cases are captured by the
single expression $p^{y_i}(1-p)^{1-y_i}$, in which the exponents switch the
factors on and off.  Hence

$$
P(\mathcal{D}\mid\bm{\theta})
   = \prod_{i=1}^{n}
     \left[p(y_i=1\mid x_i,\bm{\theta})\right]^{y_i}
     \left[1-p(y_i=1\mid x_i,\bm{\theta})\right]^{1-y_i} .\tag{5.11}
$$

This is a product of Bernoulli probabilities, exactly as
Eq. (3.18) was a product of Gaussians.

As in Section *Deriving least squares from a probability distribution* we take logarithms, both because it
turns the product into a sum and because a product of $n$ numbers smaller
than one underflows.  The log-likelihood is

$$
\log P(\mathcal{D}\mid\bm{\theta})
   = \sum_{i=1}^{n}\Big(
       y_i\log p(y_i=1\mid x_i,\bm{\theta})
       + (1-y_i)\log\left[1-p(y_i=1\mid x_i,\bm{\theta})\right]\Big).\tag{5.12}
$$

**The cost function.** 
Since we minimise rather than maximise, the cost is the negative
log-likelihood,

$$
\boxed{\;
  C(\bm{\theta}) = -\sum_{i=1}^{n}\Big(
    y_i\log p_i + (1-y_i)\log\left[1-p_i\right]\Big),
  \qquad p_i = \sigma(\theta_0+\theta_1x_i) . \;}\tag{5.13}
$$

This is known in statistics and information theory as the *cross
entropy*, and it is the loss function used to train essentially every
classifier in this book.

Substituting Eq. (5.8) and reordering the logarithms gives a form
which is both more compact and numerically better behaved.  Using
$\log p_i = (\theta_0+\theta_1x_i)-\log[1+\exp(\theta_0+\theta_1x_i)]$ and
$\log(1-p_i)=-\log[1+\exp(\theta_0+\theta_1x_i)]$,

$$
C(\bm{\theta}) = -\sum_{i=1}^{n}
    \Big(y_i(\theta_0+\theta_1x_i)
      - \log\left[1+\exp(\theta_0+\theta_1x_i)\right]\Big).\tag{5.14}
$$

**Convexity.** 
The cross entropy is a convex function of $\bm{\theta}$, so by
Section *Convexity* any local minimiser is a global minimiser.  We
verify this in Section *Gradients, the Hessian and convexity* by exhibiting the Hessian and
showing it is positive semi-definite.  This is a substantial guarantee, and it
is why logistic regression is a well-behaved problem in a way that neural
networks are not: we may apply any of the methods of
Chapter 4 without worrying about which minimum we reach.

Finally, note that just as in linear regression we often supplement the cross
entropy with regularisation terms, usually the $\ell_1$ and $\ell_2$ penalties
of Sections *Ridge regression* and *The Lasso*; we return to this in
Section *More predictors, and regularisation*.

```{admonition} Why not squared error?
:class: tip
It is natural to ask why we do not simply
minimise $\sum_i(y_i-p_i)^{2}$, which is also well defined.  There are two
answers.  The statistical one is that the squared error is the
maximum-likelihood loss for *Gaussian* noise, which is the wrong model
for a binary outcome; the cross entropy is the correct one for a Bernoulli
outcome, exactly as promised in the notebox of
Section *Deriving least squares from a probability distribution*.  The practical one is about gradients.
Differentiating the squared error brings down a factor
$\sigma'(t)=\sigma(1-\sigma)$ by Eq. (5.5), which is
*small* whenever the prediction is confident, whether or not it is
correct.  A badly wrong, confidently made prediction therefore produces almost
no gradient and the model learns nothing from its worst mistakes.  The
cross-entropy gradient, as the next section shows, contains no such factor: it
is simply the error $y_i-p_i$.  This cancellation is one of the reasons the
cross entropy is universal in classification.
```


## Gradients, the Hessian and convexity

Minimising Eq. (5.14) with respect to the two
parameters, and using
$\exp(t)/[1+\exp(t)]=\sigma(t)=p_i$, gives

$$
\begin{align}
\frac{\partial C(\bm{\theta})}{\partial\theta_0}
   &= -\sum_{i=1}^{n}\left(y_i - p_i\right),
  \\
  \frac{\partial C(\bm{\theta})}{\partial\theta_1}
   &= -\sum_{i=1}^{n}\left(y_ix_i - x_ip_i\right)
    = -\sum_{i=1}^{n}x_i\left(y_i-p_i\right).
\end{align}
$$

Both have the same structure: the residual $y_i-p_i$ weighted by the
corresponding feature, with the intercept carrying the implicit feature $1$.

**Compact form.** 
Define the vector $\bm{y}$ of the $n$ labels, the $n\times p$ design matrix
$\bm{X}$ containing the inputs including a column of ones, and the vector
$\bm{p}$ of fitted probabilities with entries
$p_i=p(y_i=1\mid\bm{x}_i,\bm{\theta})$.  Then
Eqs. (5.15) and (5.16) collapse to

$$
\boxed{\;
  \frac{\partial C(\bm{\theta})}{\partial\bm{\theta}}
   = -\bm{X}^{T}\left(\bm{y}-\bm{p}\right) . \;}\tag{5.17}
$$

This should look extremely familiar.  Compare it with the least-squares
gradient of Eq. (1.38),
$-\tfrac{2}{n}\bm{X}^{T}(\bm{y}-\bm{X}\bm{\theta})$.  The two are identical in
form, differing only in that the linear prediction $\bm{X}\bm{\theta}$ has
been replaced by the probability $\bm{p}=\sigma(\bm{X}\bm{\theta})$ and in an
overall constant.  In both cases the gradient is the design matrix transposed
against the residual, and the algorithmic remark of the notebox in
Section *The Hessian matrix* applies unchanged: the gradient costs two
matrix-vector products and never requires a matrix to be formed.

The similarity is not superficial.  Both models are *generalised linear
models*: a linear predictor $\eta_i=\bm{X}_{i,\ast}\bm{\theta}$ is passed
through a link function, the identity in one case and the sigmoid in the
other, and for the canonical link the maximum-likelihood gradient always takes
the form "design matrix transposed times residual".

**The Hessian.** 
Differentiating Eq. (5.17) once more requires the derivative of
$\bm{p}$ with respect to $\bm{\theta}$, which by
Eq. (5.5) and the chain rule (1.50) is
$\partial p_i/\partial\bm{\theta}=p_i(1-p_i)\bm{X}_{i,\ast}$.  Defining the
diagonal matrix $\bm{W}$ with entries

$$
W_{ii} = p_i\left(1-p_i\right),\tag{5.18}
$$

we obtain the compact expression

$$
\boxed{\;
  \frac{\partial^{2}C(\bm{\theta})}{\partial\bm{\theta}\,\partial\bm{\theta}^{T}}
   = \bm{X}^{T}\bm{W}\bm{X} . \;}\tag{5.19}
$$

Compare again with least squares, where the Hessian was $\bm{X}^{T}\bm{X}$ by
Eq. (1.44).  Logistic regression inserts a diagonal weight
matrix between the two factors, and the weights are largest, equal to
$\tfrac14$, where $p_i=\tfrac12$ -- that is for the observations the model is
most uncertain about, those nearest the decision boundary.  Points far from
the boundary have $p_i$ near $0$ or $1$, hence $W_{ii}$ near zero, and
contribute almost nothing to the curvature.  *The fit is determined by
the points near the boundary*, which is a first hint of the idea carried to
its conclusion by support vector machines.

**Convexity, proved.** 
For any vector $\bm{z}$,

$$
\bm{z}^{T}\bm{X}^{T}\bm{W}\bm{X}\bm{z}
   = \left(\bm{X}\bm{z}\right)^{T}\bm{W}\left(\bm{X}\bm{z}\right)
   = \sum_{i=1}^{n}W_{ii}\left(\bm{X}\bm{z}\right)_i^{2} \ \ge\ 0,\tag{5.20}
$$

because every $W_{ii}=p_i(1-p_i)$ is positive for $0<p_i<1$.  The Hessian is
therefore positive semi-definite everywhere, so by the second-order condition
of Section *Convexity* the cross entropy is convex and by the result
quoted there any stationary point is a global minimum.  This is the promised
proof.

**No closed form.** 
Unlike the normal equations (1.39), setting
Eq. (5.17) to zero does not yield a formula for
$\hat{\bm{\theta}}$: the unknown appears inside the non-linear function
$\bm{p}=\sigma(\bm{X}\bm{\theta})$, and no rearrangement extracts it.  We must
solve numerically, which is why Chapter 4 preceded this one.
The situation resembles the Lasso of Section *The Lasso* -- a convex
problem without a closed form -- although here the obstruction is
non-linearity rather than non-differentiability, so ordinary gradient methods
apply directly.


## More predictors, and regularisation

Extending to $p$ predictors requires no new ideas.  The log-odds
relation (5.10) becomes

$$
\ln\frac{p(\bm{\theta},\bm{x})}{1-p(\bm{\theta},\bm{x})}
   = \theta_0 + \theta_1x_1 + \theta_2x_2 + \dots + \theta_px_p,\tag{5.21}
$$

and with $\bm{x}=[1,x_1,\dots,x_p]$ and
$\bm{\theta}=[\theta_0,\theta_1,\dots,\theta_p]$ the probability is

$$
p(\bm{\theta},\bm{x})
   = \frac{\exp\left(\theta_0+\theta_1x_1+\dots+\theta_px_p\right)}
          {1+\exp\left(\theta_0+\theta_1x_1+\dots+\theta_px_p\right)}
   = \sigma\!\left(\bm{x}^{T}\bm{\theta}\right).\tag{5.22}
$$

The gradient (5.17) and Hessian (5.19) are
already written in a form valid for any $p$.

**Regularisation.** 
In practice the cross entropy is supplemented with a penalty, exactly as in
Chapter 3,

$$
C_{\lambda}(\bm{\theta}) = C(\bm{\theta})
    + \lambda\left\|\bm{\theta}\right\|_2^{2}
  \qquad\text{or}\qquad
  C_{\lambda}(\bm{\theta}) = C(\bm{\theta})
    + \lambda\left\|\bm{\theta}\right\|_1,\tag{5.23}
$$

giving the $\ell_2$ and $\ell_1$ regularised versions.  The gradient of the
$\ell_2$ penalty is $2\lambda\bm{\theta}$ by Eq. (1.28), so
Eq. (5.17) becomes
$-\bm{X}^{T}(\bm{y}-\bm{p})+2\lambda\bm{\theta}$, and the Hessian becomes
$\bm{X}^{T}\bm{W}\bm{X}+2\lambda\bm{I}$, which is positive *definite* for
$\lambda>0$ and hence strictly convex.

Everything said in Chapter 3 about the two penalties carries
over: $\ell_2$ shrinks coefficients smoothly, $\ell_1$ sets some exactly to
zero and performs variable selection, the intercept should not be penalised,
and the features must be standardised first for the penalty to be meaningful.
The remarks of Section *Scaling, centring and the intercept* apply verbatim.

There is one addition specific to classification.  If the two classes are
*linearly separable* -- if some hyperplane divides them perfectly -- then
the unregularised maximum likelihood problem has *no finite solution*.
The likelihood can always be increased by scaling $\bm{\theta}$ up, which
makes every predicted probability more extreme and drives the cost towards
zero without ever attaining it, so $\|\bm{\theta}\|\to\infty$.  Any penalty
$\lambda>0$ removes the pathology by making the cost eventually increase.
This is one reason `scikit-learn` regularises by default, with
`C` playing the role of $1/\lambda$ -- a convention worth remembering
when comparing against one's own implementation, in the spirit of the
warnings in Section *Comparing the three estimators*.

```{admonition} Machine learning connection
:class: tip
The conditioning discussion of
Section *The learning rate and the condition number* applies here with a twist.  The Hessian is now
$\bm{X}^{T}\bm{W}\bm{X}$, which *changes as the fit proceeds* because
$\bm{W}$ depends on the current probabilities.  Early in training, when all
$p_i\approx\tfrac12$ and $\bm{W}\approx\tfrac14\bm{I}$, the curvature is
essentially $\tfrac14\bm{X}^{T}\bm{X}$ and the analysis of
Chapter 4 carries over directly.  As the fit sharpens, the
well-classified points drop out of $\bm{W}$ and the effective Hessian is built
from a shrinking subset of the data, typically becoming worse conditioned.  A
learning rate chosen at the start may therefore be badly wrong later, which is
one concrete reason the adaptive methods of Section *Why adapt the step size at all* --
whose $\bm{v}_t$ tracks a *recent* average and so follows the changing
landscape -- are useful even on this convex problem.
```


## More than two classes: the softmax

Suppose we wish to extend to $K$ classes.  Taking one class as a reference --
conventionally the last -- we model each of the remaining $K-1$ log-odds
ratios as linear.  With a single predictor for simplicity,

$$
\begin{align}
\ln\frac{p(C=1\mid x)}{p(C=K\mid x)} &= \theta_{10}+\theta_{11}x_1,
    \nonumber\\
  \ln\frac{p(C=2\mid x)}{p(C=K\mid x)} &= \theta_{20}+\theta_{21}x_1,
    \nonumber\\
  &\ \ \vdots \nonumber\\
  \ln\frac{p(C=K-1\mid x)}{p(C=K\mid x)}
    &= \theta_{(K-1)0}+\theta_{(K-1)1}x_1 .
\end{align}
$$

The model is thus specified by $K-1$ logit transformations.  Solving for the
probabilities, and using the fact that they must sum to one,

$$
p(C=k\mid\bm{x}) =
    \frac{\exp\left(\theta_{k0}+\theta_{k1}x_1\right)}
         {1+\sum_{l=1}^{K-1}\exp\left(\theta_{l0}+\theta_{l1}x_1\right)},
  \qquad k=1,\dots,K-1,\tag{5.25}
$$

with the reference class taking the remainder,

$$
p(C=K\mid\bm{x}) =
    \frac{1}{1+\sum_{l=1}^{K-1}\exp\left(\theta_{l0}+\theta_{l1}x_1\right)} .\tag{5.26}
$$

These sum to one by construction, and setting $K=2$ recovers
Eqs. (5.8) and (5.9), so the binary case is the special
case it should be.  Extending to more predictors requires only replacing
$\theta_{k0}+\theta_{k1}x_1$ by $\bm{x}^{T}\bm{\theta}_k$.

**The symmetric form.** 
Singling out a reference class is inelegant, and in practice one uses the
symmetric parametrisation

$$
\boxed{\;
  p(C=k\mid\bm{x}) = \frac{\exp\left(\bm{x}^{T}\bm{\theta}_k\right)}
    {\sum_{l=0}^{K-1}\exp\left(\bm{x}^{T}\bm{\theta}_l\right)},
  \qquad k=0,\dots,K-1, \;}\tag{5.27}
$$

which is the *softmax* function.  It is over-parametrised: adding the
same vector to every $\bm{\theta}_k$ leaves all the probabilities unchanged,
since the added factor cancels between numerator and denominator, so the
parameters are determined only up to a common shift.  Fixing
$\bm{\theta}_{K-1}=\bm{0}$ recovers Eq. (5.25), and adding an
$\ell_2$ penalty also removes the redundancy by selecting the smallest
representative.  The redundancy is harmless in practice and the symmetric form
is easier to implement.

The corresponding cost function is the multiclass cross entropy,

$$
C(\bm{\Theta}) = -\sum_{i=1}^{n}\sum_{k=0}^{K-1}
    y_{ik}\log p(C=k\mid\bm{x}_i),\tag{5.28}
$$

where $y_{ik}$ is the *one-hot* encoding, equal to one if sample $i$
belongs to class $k$ and zero otherwise.  For $K=2$ this reduces to
Eq. (5.13).  Its gradient with respect to
$\bm{\theta}_k$ retains the familiar structure,

$$
\frac{\partial C}{\partial\bm{\theta}_k}
   = -\bm{X}^{T}\left(\bm{y}_{k}-\bm{p}_{k}\right),\tag{5.29}
$$

with $\bm{y}_k$ and $\bm{p}_k$ the columns of the one-hot targets and of the
predicted probabilities for class $k$ -- again design matrix transposed
against residual.

**A numerical warning.** 
Equation (5.27) must never be evaluated as written.  If any
$\bm{x}^{T}\bm{\theta}_k$ is large, $\exp$ of it overflows.  Since the softmax
is invariant under subtracting a constant from every score, one subtracts the
largest before exponentiating,

$$
p_k = \frac{\exp\left(s_k-s_{\max}\right)}
             {\sum_l\exp\left(s_l-s_{\max}\right)},
  \qquad s_k=\bm{x}^{T}\bm{\theta}_k,\tag{5.30}
$$

which is mathematically identical and numerically safe: every exponent is now
non-positive, so no term exceeds one.  This trick appears in the
implementation of Section *An implementation* and in every deep learning
library.

The softmax will reappear in the next chapter as the output layer of a
classification network, where Eq. (5.27) applied to the final
linear layer is exactly multinomial logistic regression performed on learned
rather than raw features.


## Optimising the cross entropy

Having no closed form, we must minimise Eq. (5.13)
numerically.  This is where Chapter 4 is spent, and the good
news is that every method there applies without modification: the problem is
convex, the gradient is Eq. (5.17), and the Hessian is
Eq. (5.19).

**Gradient descent.** 
The simplest scheme is Eq. (4.15) with the logistic
gradient,

$$
\bm{\theta}_{k+1} = \bm{\theta}_k
    + \eta\,\bm{X}^{T}\left(\bm{y}-\bm{p}(\bm{\theta}_k)\right),\tag{5.31}
$$

noting the plus sign, which comes from the minus in
Eq. (5.17).  The stability
bound (4.20) applies with $\lambda_{\max}$ the largest
eigenvalue of $\bm{X}^{T}\bm{W}\bm{X}$; since $W_{ii}\le\tfrac14$, a safe
choice is $\eta<8/\lambda_{\max}(\bm{X}^{T}\bm{X})$.  For large data sets one
uses the minibatch stochastic version of Section *Stochastic gradient descent*, and in
practice one of the adaptive methods of Sections *AdaGrad* to
*Adam*.

**Newton-Raphson, or iteratively reweighted least squares.** 
Because the Hessian is available in the closed
form (5.19), and because $p$ is often modest, Newton's method of
Section *Newton's method* is frequently the method of choice here -- one of the
few places in this book where it is affordable.  Substituting
Eqs. (5.17) and (5.19) into the Newton
step (4.8),

$$
\bm{\theta}^{\mathrm{new}} = \bm{\theta}^{\mathrm{old}}
   - \left(\frac{\partial^{2}C}
      {\partial\bm{\theta}\partial\bm{\theta}^{T}}\right)^{-1}_{\bm{\theta}^{\mathrm{old}}}
     \left(\frac{\partial C}{\partial\bm{\theta}}\right)_{\bm{\theta}^{\mathrm{old}}},\tag{5.32}
$$

that is, in matrix form,

$$
\boxed{\;
  \bm{\theta}^{\mathrm{new}} = \bm{\theta}^{\mathrm{old}}
   + \left(\bm{X}^{T}\bm{W}\bm{X}\right)^{-1}
     \bm{X}^{T}\left(\bm{y}-\bm{p}\right), \;}\tag{5.33}
$$

with the right-hand side evaluated at the old parameters.  Unlike the
quadratic case of Eq. (4.9), one step does not suffice, since
$\bm{W}$ and $\bm{p}$ must be recomputed; but convergence is quadratic and a
handful of iterations is typical.

Equation (5.33) has an illuminating rewriting.  Defining the
*adjusted response*
$\bm{z}=\bm{X}\bm{\theta}^{\mathrm{old}}+\bm{W}^{-1}(\bm{y}-\bm{p})$, a line
of algebra turns it into

$$
\bm{\theta}^{\mathrm{new}}
   = \left(\bm{X}^{T}\bm{W}\bm{X}\right)^{-1}\bm{X}^{T}\bm{W}\bm{z},\tag{5.34}
$$

which is precisely the *weighted least squares*
solution (3.13) with weights $\bm{W}$ and targets $\bm{z}$.
Logistic regression is therefore solved by repeatedly performing a weighted
linear regression, recomputing the weights at each pass -- whence the name
*iteratively reweighted least squares* (IRLS).  It is a pleasing
connection: the classifier reduces to a sequence of the linear fits of
Chapter 3.

In practice one does not form the inverse in Eq. (5.33).  Since
$\bm{X}^{T}\bm{W}\bm{X}$ is symmetric positive semi-definite, the step is
obtained from a Cholesky factorisation as in Section *LU and Cholesky decompositions*, or more
safely still from the QR or SVD route of Section *Ordinary least squares* applied to
$\bm{W}^{1/2}\bm{X}$, which avoids squaring the condition number.


In [ ]:
import numpy as np

def sigmoid(z):
    """Numerically stable logistic function, Eq. (5.sigmoid)."""
    out = np.empty_like(z, dtype=float)
    pos, neg = z >= 0, z < 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[neg])                       # avoids overflow for z << 0
    out[neg] = ez / (1.0 + ez)
    return out


def logreg_newton(X, y, n_iter=25, tol=1e-10, ridge=1e-8):
    """Logistic regression by Newton-Raphson, Eq. (5.newton).

    A tiny ridge term keeps X^T W X invertible when the classes are
    separable or W becomes numerically singular.
    """
    n, p = X.shape
    theta = np.zeros(p)
    for it in range(n_iter):
        prob = sigmoid(X @ theta)
        gradient = X.T @ (y - prob)                       # Eq. (5.gradient)
        W = prob * (1.0 - prob)                           # Eq. (5.Wmatrix)
        H = X.T @ (W[:, None] * X) + ridge * np.eye(p)    # Eq. (5.hessian)
        step = np.linalg.solve(H, gradient)               # never invert H
        theta += step
        if np.linalg.norm(step) < tol:
            break
    return theta, it + 1


Note the stable sigmoid.  Evaluating $1/(1+e^{-z})$ directly overflows for
large negative $z$, and the two-branch form above is the standard remedy --
the same defensive reasoning as the softmax shift of
Eq. (5.30).

```{admonition} Machine learning connection
:class: tip
Which optimiser to use is decided by the
shape of the problem, and logistic regression makes the trade-off unusually
clear.  Newton's method costs $\bigO(np^{2})$ to form the Hessian and
$\bigO(p^{3})$ to factorise it, but converges in a handful of iterations; for
$p$ in the tens or hundreds it is unbeatable, and it is what
`scikit-learn` uses through its `lbfgs` and `newton-cg`
solvers.  For $p$ in the millions -- text classification with a bag-of-words
representation, say -- the Hessian is unavailable and one falls back on SGD
with an adaptive method.  The dividing line is exactly the one drawn in
Section *None of these can compete with Newton's method*, and logistic regression sits close enough to
it that both sides are commonly seen.
```


## An implementation

We now assemble a class handling both the binary and the multiclass case,
trained by gradient descent.  It is deliberately written to mirror the
equations: the reader should be able to point at each line and name the
formula it implements.


In [ ]:
import numpy as np

class LogisticRegression:
    """Logistic regression for binary and multiclass classification.

    Binary problems use the sigmoid (5.sigmoid) with the cross entropy
    (5.crossentropy); multiclass problems use the softmax (5.softmax)
    with the multiclass cross entropy (5.multicrossentropy).
    """

    def __init__(self, lr=0.01, epochs=1000, fit_intercept=True, lmbda=0.0):
        self.lr = lr                       # learning rate for gradient descent
        self.epochs = epochs
        self.fit_intercept = fit_intercept
        self.lmbda = lmbda                 # l2 penalty, Eq. (5.penalised)
        self.weights = None
        self.multi_class = False

    @staticmethod
    def _add_intercept(X):
        return np.concatenate((np.ones((X.shape[0], 1)), X), axis=1)

    @staticmethod
    def _sigmoid(z):
        out = np.empty_like(z, dtype=float)
        pos, neg = z >= 0, z < 0
        out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
        ez = np.exp(z[neg])
        out[neg] = ez / (1.0 + ez)
        return out

    @staticmethod
    def _softmax(Z):
        """Softmax with the shift of Eq. (5.softmaxstable)."""
        expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return expZ / np.sum(expZ, axis=1, keepdims=True)

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        if self.fit_intercept:
            X = self._add_intercept(X)
        n_samples, n_features = X.shape

        self.classes_ = np.unique(y)
        self.multi_class = len(self.classes_) > 2

        if self.multi_class:
            n_classes = len(self.classes_)
            index = {c: k for k, c in enumerate(self.classes_)}
            Y = np.zeros((n_samples, n_classes))            # one-hot targets
            Y[np.arange(n_samples), [index[c] for c in y]] = 1
            self.weights = np.zeros((n_features, n_classes))

            for _ in range(self.epochs):
                probs = self._softmax(X @ self.weights)
                grad = X.T @ (probs - Y) / n_samples        # Eq. (5.softmaxgradient)
                grad += 2.0 * self.lmbda * self.weights
                self.weights -= self.lr * grad
        else:
            yb = (y == self.classes_[1]).astype(float)      # map labels to {0,1}
            self.weights = np.zeros(n_features)

            for _ in range(self.epochs):
                probs = self._sigmoid(X @ self.weights)
                grad = X.T @ (probs - yb) / n_samples       # Eq. (5.gradient)
                grad += 2.0 * self.lmbda * self.weights
                self.weights -= self.lr * grad
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        if self.fit_intercept:
            X = self._add_intercept(X)
        if self.multi_class:
            return self._softmax(X @ self.weights)
        p1 = self._sigmoid(X @ self.weights)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


Note that the gradients carry a factor $1/n$ absent from
Eq. (5.17).  This merely rescales the learning rate and is the
usual convention, since it makes a sensible $\eta$ independent of the size of
the data set; but it is one more of the bookkeeping factors that must be
matched when comparing implementations.

**Synthetic data.** 
To exercise the class we generate Gaussian clusters, one per class.


In [ ]:
import numpy as np

def generate_binary_data(n_samples=100, n_features=2, random_state=None):
    """Two Gaussian clusters, class 0 around -2 and class 1 around +2."""
    rng = np.random.default_rng(random_state)
    n0 = n_samples // 2
    n1 = n_samples - n0
    X0 = rng.normal(size=(n0, n_features)) - 2.0
    X1 = rng.normal(size=(n1, n_features)) + 2.0
    return np.vstack((X0, X1)), np.array([0] * n0 + [1] * n1)


def generate_multiclass_data(n_samples=150, n_features=2, n_classes=3,
                             random_state=None):
    """One Gaussian cluster per class, centred on a circle."""
    rng = np.random.default_rng(random_state)
    per = n_samples // n_classes
    Xs, ys = [], []
    for k in range(n_classes):
        angle = 2.0 * np.pi * k / n_classes
        centre = 4.0 * np.array([np.cos(angle), np.sin(angle)])
        centre = np.resize(centre, n_features)
        Xs.append(rng.normal(size=(per, n_features)) + centre)
        ys.append(np.full(per, k))
    return np.vstack(Xs), np.concatenate(ys)


X, y = generate_binary_data(200, random_state=2024)
model = LogisticRegression(lr=0.1, epochs=2000).fit(X, y)
print("binary training accuracy:", np.mean(model.predict(X) == y))

Xm, ym = generate_multiclass_data(300, n_classes=3, random_state=2024)
multi = LogisticRegression(lr=0.1, epochs=2000).fit(Xm, ym)
print("multiclass training accuracy:", np.mean(multi.predict(Xm) == ym))


Both problems are separable by construction, so both accuracies are close to
unity.  This is the moment to recall the warning of
Section *More predictors, and regularisation*: on perfectly separable data the unpenalised
likelihood has no finite maximiser, and the weights grow without bound as
training continues.  With a fixed number of epochs the growth is simply
truncated; setting `lmbda` to a small positive value is the principled
fix.


## Measuring the quality of a classifier

The mean squared error and $R^{2}$ of Section *Measures of quality* are
meaningless for a classifier.  We need measures appropriate to discrete
outcomes, and it turns out that no single number will do.

**The confusion matrix.** 
Everything starts here.  For two classes, cross-tabulating the true label
against the predicted one gives four counts:

$$
\begin{array}{c|cc}
     & \text{predicted } 1 & \text{predicted } 0\\
    \hline
    \text{true } 1 & \mathrm{TP} & \mathrm{FN}\\
    \text{true } 0 & \mathrm{FP} & \mathrm{TN}
  \end{array}\tag{5.35}
$$

the true positives, false negatives, false positives and true negatives.
Every scalar measure below is a function of these four numbers, and reporting
the matrix itself is almost always more informative than reporting any of
them.

**Accuracy, and why it misleads.** 
The obvious measure is the fraction correctly classified,

$$
\mathrm{accuracy} = \frac{\mathrm{TP}+\mathrm{TN}}
    {\mathrm{TP}+\mathrm{TN}+\mathrm{FP}+\mathrm{FN}}
   = \frac{1}{n}\sum_{i=1}^{n} I\!\left(y_i=\tilde{y}_i\right),\tag{5.36}
$$

with $I$ the indicator function.  Accuracy is the right measure when the
classes are balanced and the two kinds of error are equally costly.  It is
badly misleading otherwise.  If one in a thousand transactions is fraudulent,
the classifier that declares everything legitimate achieves $99.9\%$ accuracy
while being entirely useless.  *Always compare an accuracy against the
proportion of the majority class*, which is the score obtained for free.

**Precision and recall.** 
Separating the two kinds of error gives

$$
\mathrm{precision} = \frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FP}},
  \qquad
  \mathrm{recall} = \frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FN}} .\tag{5.37}
$$

Precision asks: of the samples flagged positive, what fraction really were?
Recall asks: of the samples that really were positive, what fraction did we
find?  The two trade off against each other through the decision threshold.
Lowering the threshold flags more samples, catching more of the true positives
-- recall rises -- at the cost of more false alarms -- precision falls.  Which
matters more is a question about the application and not about the model: in
screening for a serious disease a false negative is far worse than a false
positive, so recall dominates; in flagging emails as spam the reverse holds.
The harmonic mean of the two,

$$
F_1 = 2\,\frac{\mathrm{precision}\cdot\mathrm{recall}}
                {\mathrm{precision}+\mathrm{recall}},\tag{5.38}
$$

is the usual compromise; the harmonic mean rather than the arithmetic one
because it is small unless *both* are large.

**ROC and AUC.** 
Because logistic regression outputs a probability rather than a decision, the
threshold is ours to choose, and a fair assessment should not depend on one
arbitrary choice.  The *receiver operating characteristic* curve plots
the true positive rate $\mathrm{TP}/(\mathrm{TP}+\mathrm{FN})$, which is the
recall, against the false positive rate
$\mathrm{FP}/(\mathrm{FP}+\mathrm{TN})$, as the threshold sweeps from one to
zero.  A perfect classifier passes through the top-left corner; a classifier
that ranks at random gives the diagonal.  The area under the curve, the
*AUC*, summarises the whole sweep in a single number, and it has a clean
interpretation: it is the probability that a randomly chosen positive sample
receives a higher score than a randomly chosen negative one.  An AUC of
$0.5$ is chance and $1.0$ is perfect.  Since AUC depends only on the ranking,
it is insensitive to class imbalance in a way accuracy is not.


In [ ]:
import numpy as np

def confusion_matrix(y_true, y_pred):
    """Return TP, FN, FP, TN for binary labels in {0, 1}."""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fn, fp, tn


def classification_report(y_true, y_pred):
    tp, fn, fp, tn = confusion_matrix(y_true, y_pred)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall else 0.0)
    return dict(accuracy=accuracy, precision=precision, recall=recall, f1=f1)


def roc_auc(y_true, scores):
    """AUC as the probability that a positive outranks a negative."""
    order = np.argsort(scores)
    ranks = np.empty(len(scores), dtype=float)
    ranks[order] = np.arange(1, len(scores) + 1)
    n_pos = int(np.sum(y_true == 1))
    n_neg = len(y_true) - n_pos
    return (np.sum(ranks[y_true == 1]) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


```{admonition} Machine learning connection
:class: tip
The cross entropy is the loss we
*optimise*; accuracy, $F_1$ and AUC are what we *report*.  They are
not the same and cannot be, because accuracy is a step function of the
parameters -- flat almost everywhere, with jumps where a prediction crosses
the threshold -- so its gradient is zero wherever it is defined and no method
of Chapter 4 could minimise it.  The cross entropy is a smooth
surrogate that is minimised in roughly the same place.  This split between a
differentiable training objective and a non-differentiable evaluation metric
runs through all of machine learning, and it is worth being explicit that a
model with the lower cross entropy does not automatically have the higher
accuracy.  It also explains why, when the classes are severely imbalanced,
one often weights the cross entropy by class frequency: to move the minimum of
the surrogate closer to the optimum of the metric one actually cares about.
```


## The Wisconsin breast cancer data

We close with a real data set.  The Wisconsin breast cancer data contain $569$
samples, each described by $30$ features computed from a digitised image of a
fine-needle aspirate of a breast mass -- radius, texture, perimeter, area,
smoothness and so on, each in three variants -- and each labelled malignant or
benign.  It is a natural test for a binary classifier and is included with
`scikit-learn`.


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)
print(X_train.shape, X_test.shape)

# Scaling inside the pipeline, so it is refitted on each training fold
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
logreg.fit(X_train, y_train)

print(f"test accuracy: {logreg.score(X_test, y_test):.3f}")
scores = cross_validate(logreg, X_train, y_train, cv=10)["test_score"]
print(f"cross-validated accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")


The scaler is not decoration.  The thirty features differ enormously in
scale -- the mean area is around $655$ while the mean smoothness is around
$0.096$, a ratio of some four orders of magnitude, and the ratio of the
largest to the smallest standard deviation across the thirty columns exceeds
$10^{5}$ -- so by Section *More predictors, and regularisation* the default $\ell_2$
penalty would fall almost entirely on the small-scale features, and by
Section *The learning rate and the condition number* the optimisation would be badly conditioned.
Placing the scaler inside a `Pipeline` ensures it is refitted on each
training fold, honouring the discipline of
Section *Cross-validation*.

**Inspecting the features.** 
Before fitting anything it is worth looking at the data.  Plotting, for each
feature, the histograms of the two classes superimposed shows immediately
which features separate them.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

cancerpd = pd.DataFrame(cancer.data, columns=cancer.feature_names)
malignant = cancer.data[cancer.target == 0]
benign = cancer.data[cancer.target == 1]

fig, axes = plt.subplots(15, 2, figsize=(10, 20))
ax = axes.ravel()
for i in range(30):
    _, bins = np.histogram(cancer.data[:, i], bins=50)
    ax[i].hist(malignant[:, i], bins=bins, alpha=0.5)
    ax[i].hist(benign[:, i], bins=bins, alpha=0.5)
    ax[i].set_title(cancer.feature_names[i])
    ax[i].set_yticks(())
ax[0].set_xlabel("Feature magnitude")
ax[0].set_ylabel("Frequency")
ax[0].legend(["Malignant", "Benign"], loc="best")
fig.tight_layout()
plt.show()

# The correlation matrix of Section 1.covariance, computed with pandas
correlations = cancerpd.corr()
plt.figure(figsize=(10, 9))
plt.imshow(correlations, vmin=-1, vmax=1, cmap="RdBu_r")
plt.colorbar(); plt.title("Correlation matrix of the features")
plt.show()


The correlation matrix repays study, and it connects directly to
Chapter 1.  Several groups of features are almost perfectly
correlated -- radius, perimeter and area are essentially the same measurement
three times over, since a roughly circular region has perimeter and area
determined by its radius.  By Section *Rank, the pseudoinverse and ill-conditioned design matrices* this makes the
design matrix nearly rank deficient, so the individual coefficients are poorly
determined even though the predictions are not.  Three remedies are available
and all appear in this book: penalise with $\ell_2$, as the default settings
do; select with $\ell_1$, which will keep one member of each correlated group;
or reduce the dimension first with the principal component analysis of
Section *Principal component analysis*.  The point to carry away is that a high accuracy on
this data set does *not* license a statement about which feature matters,
because the data cannot distinguish the members of a correlated group.


In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, RocCurveDisplay)

y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred,
                            target_names=cancer.target_names))
print(f"AUC: {roc_auc_score(y_test, y_proba):.4f}")

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.show()


The run above gives a test accuracy of $0.958$, a cross-validated accuracy on
the training set of $0.981\pm0.025$, and an AUC of $0.9914$.  The confusion
matrix on the $143$ test samples is

$$
\begin{bmatrix} 50 & 3\\ 3 & 87\end{bmatrix},
$$

with rows the true class (malignant, benign) and columns the predicted one.

These numbers should be read with the cautions of
Section *Measuring the quality of a classifier* in mind.  An accuracy of $0.958$ sounds
impressive until one notices that $62.7\%$ of the samples are benign, so the
classifier that predicts "benign" unconditionally already scores $0.629$;
the model has removed about nine tenths of the remaining error, which is the
honest way to state the achievement.  The AUC of $0.9914$ is the more
informative summary, being independent of both the threshold and the class
balance.

The confusion matrix is more informative still, because it separates the two
kinds of error: three malignant cases were predicted benign and three benign
cases malignant.  In a screening context those are not equivalent, and one
would deliberately move the threshold below $0.5$ to reduce the first at the
cost of increasing the second -- trading precision for recall, exactly as
discussed in Section *Measuring the quality of a classifier*.  Note finally that the
cross-validated accuracy ($0.981$) exceeds the test accuracy ($0.958$); with
$143$ test samples the standard error of the latter is about $0.017$ by
Eq. (2.24), so the difference is within about one and a half
standard errors and should not be over-interpreted.


## Summary and the programs

Logistic regression is the bridge between the linear models of
Chapter 3 and the neural networks of the next chapter, and it is
worth setting out how little had to change and how much followed.

The model changed by one function.  We kept the linear predictor
$\bm{x}^{T}\bm{\theta}$ and passed it through the sigmoid (5.3),
which confines the output to $(0,1)$ so that it can be read as a probability.
Equivalently, by Eq. (5.10), we modelled the log-odds as linear
while the probability itself is not.

The loss followed from the model rather than being chosen.  Repeating the
maximum-likelihood argument of Section *Deriving least squares from a probability distribution* with Bernoulli
rather than Gaussian observations produced the cross
entropy (5.13), just as the notebox there predicted.  This
is worth remembering as a general principle: specify the distribution of the
target and the loss function is determined.

The mathematics then ran parallel to Chapter 3 throughout.  The
gradient is $-\bm{X}^{T}(\bm{y}-\bm{p})$, design matrix against residual,
exactly as in least squares with the linear prediction replaced by the
probability.  The Hessian is $\bm{X}^{T}\bm{W}\bm{X}$ rather than
$\bm{X}^{T}\bm{X}$, with weights $p_i(1-p_i)$ that vanish for confidently
classified points -- so the fit is controlled by the points near the decision
boundary.  Positivity of those weights proves convexity, so
Chapter 4 applies with its guarantees intact.  What was lost is
the closed form: the parameter appears inside a non-linear function and cannot
be extracted, so we optimise numerically, by gradient descent for large
problems and by Newton-Raphson -- which is iteratively reweighted least
squares, Eq. (5.34) -- for small ones.

Two extensions completed the picture.  The softmax (5.27)
generalises the sigmoid to $K$ classes and preserves the gradient structure,
and the penalties of Chapter 3 carry over unchanged, with the
additional observation that some penalty is *necessary* when the classes
are linearly separable.  Finally, assessing a classifier needs a different
vocabulary from assessing a regression: the confusion matrix and the measures
built from it, with the standing warning that accuracy alone is uninformative
when the classes are imbalanced.

The complete programs are collected in `doc/BookML/BookPrograms`:

- `logistic_regression.py` -- the class of
   Section *An implementation* for the binary and multiclass cases, the
   synthetic data generators, and the numerically stable sigmoid and
   softmax of Eqs. (5.3) and (5.30).
- `logreg_newton.py` -- Newton-Raphson and IRLS as in
   Eqs. (5.33) and (5.34), with a comparison of
   iteration counts against gradient descent and against the adaptive
   methods of Chapter 4.
- `classification_metrics.py` -- the confusion matrix, accuracy,
   precision, recall, $F_1$ and AUC of Section *Measuring the quality of a classifier*,
   checked against `scikit-learn`.
- `wisconsin.py` -- the complete analysis of
   Section *The Wisconsin breast cancer data*, including the feature histograms, the
   correlation matrix, the cross-validated accuracy and the ROC curve.

Each file runs as a script and reproduces the numbers quoted in this chapter.
An executable version is available as a Jupyter notebook in the accompanying
Jupyter-book.

**Looking ahead.** 
Logistic regression is a neural network with no hidden layer.  Its linear
predictor is a single artificial neuron, its sigmoid is that neuron's
activation function, and its cross entropy is the loss used to train
classification networks.  The next chapter inserts layers between the input and
the output, so that the features fed to the final sigmoid are themselves
learned rather than given; the gradient of the last layer will be exactly
Eq. (5.17), and backpropagation is the chain
rule (1.51) carrying that gradient backwards through the
preceding layers.  Almost everything in this chapter will be reused.


## Exercises

### Warm-up exercises

1. **Properties of the sigmoid.**
   (a) Prove Eq. (5.4), $1-\sigma(t)=\sigma(-t)$.
   (b) Prove Eq. (5.5),
   $\sigma'=\sigma(1-\sigma)$, and show that the derivative attains its
   maximum value $1/4$ at $t=0$.
   (c) Verify Eq. (5.7), $\tanh(t)=2\sigma(2t)-1$, and deduce the
   derivative of $\tanh$ in terms of $\tanh$ itself.
   (d) Show that the logit (5.6) is the inverse of the sigmoid.
2. **The cross entropy from the likelihood.**
   (a) Explain why the single expression $p^{y_i}(1-p)^{1-y_i}$ correctly
   handles both $y_i=0$ and $y_i=1$.
   (b) Derive Eq. (5.13) from
   Eq. (5.11).
   (c) Derive the compact form (5.14) and explain why
   it is numerically preferable.
3. **The gradient.**
   (a) Derive Eqs. (5.15) and (5.16) by
   differentiating Eq. (5.14).
   (b) Assemble them into the matrix form (5.17).
   (c) Compare with the least-squares gradient (1.38) and
   state precisely what is the same and what differs.
4. **Why not squared error (numerical).**
   Consider one observation with $y=1$ and a confidently wrong prediction,
   $\sigma(t)=0.01$.
   (a) Compute the gradient with respect to $t$ of the squared error
   $(y-\sigma(t))^{2}$ and of the cross entropy.
   (b) Repeat for $\sigma(t)=0.5$ and $\sigma(t)=0.99$.
   (c) Plot both gradients against $t$ and explain, using the notebox of
   Section *Maximum likelihood and the cross-entropy*, why the cross entropy is preferred.
5. **The Hessian and convexity.**
   (a) Derive Eq. (5.19) from Eq. (5.17).
   (b) Prove positive semi-definiteness as in Eq. (5.20).
   (c) Show that adding an $\ell_2$ penalty makes the Hessian positive
   definite, and explain why the problem is then strictly convex.
   (d) Show that $W_{ii}\le 1/4$ and use this to justify the learning-rate
   bound quoted in Section *Optimising the cross entropy*.
6. **Separability (numerical).**
   Generate two classes that are perfectly separable in one dimension.
   (a) Fit an unpenalised logistic regression by gradient descent and plot
   $\|\bm{\theta}\|$ against the iteration count.  What happens?
   (b) Plot the cost against the iteration count.  Does it converge?
   (c) Repeat with an $\ell_2$ penalty and explain the difference.
7. **Softmax.**
   (a) Show that Eq. (5.27) reduces to
   Eqs. (5.8) and (5.9) when $K=2$ and
   $\bm{\theta}_0=\bm{0}$.
   (b) Show that adding a constant vector to every $\bm{\theta}_k$ leaves the
   probabilities unchanged.
   (c) Show that Eq. (5.30) is mathematically identical to
   Eq. (5.27), and construct a numerical example in which the
   naive form overflows.
8. **Newton-Raphson and IRLS (numerical).**
   (a) Implement Eq. (5.33) and verify that it converges in a
   handful of iterations on the synthetic data of
   Section *An implementation*.
   (b) Verify algebraically that Eq. (5.33) can be rewritten as
   Eq. (5.34) with the adjusted response
   $\bm{z}=\bm{X}\bm{\theta}+\bm{W}^{-1}(\bm{y}-\bm{p})$.
   (c) Compare the iteration count against plain gradient descent and against
   Adam from Section *Adam*, and discuss when each is preferable.
9. **Metrics (numerical).**
   For a deliberately imbalanced problem with $5\%$ positives:
   (a) compute the accuracy of the classifier that always predicts the majority
   class;
   (b) fit a logistic regression and compute accuracy, precision, recall,
   $F_1$ and AUC;
   (c) sweep the decision threshold from $0$ to $1$, plot precision and recall
   against it, and identify the threshold maximising $F_1$;
   (d) verify your `roc_auc` implementation against
   `sklearn.metrics.roc_auc_score`.
10. **Correlated features.**
   In the Wisconsin data of Section *The Wisconsin breast cancer data*:
   (a) find the pairs of features with correlation above $0.95$;
   (b) fit logistic regression with $\ell_2$ and with $\ell_1$ penalties and
   compare which members of each correlated group survive;
   (c) relate what you find to the discussion of the Lasso and correlated
   predictors in the notebox of Section *The Lasso*.

### Project-style exercise: classification from end to end

**Part a: your own implementation.** 
Write logistic regression from scratch, with the stable sigmoid, the
cross-entropy cost (5.13), the analytical
gradient (5.17) and an $\ell_2$ penalty.  Verify the gradient
against a finite difference using Eq. (4.43) and against
automatic differentiation as in Section *Automatic differentiation*.

**Part b: optimisers.** 
Train it on the Wisconsin data using plain gradient descent, momentum,
minibatch SGD, RMSProp and Adam from Chapter 4.  For each,
sweep the learning rate and record the best cross-entropy achieved and the
number of epochs needed.  Present the result as a table like
Table 4.1, and discuss it in the light of the warning there
that learning rates are not comparable across optimisers.

**Part c: Newton-Raphson.** 
Add the Newton solver of Eq. (5.33) and compare its iteration
count with the results of part b.  Measure the wall-clock time as well, and
determine the number of features at which the $\bigO(p^{3})$ factorisation
ceases to be worthwhile.

**Part d: model selection.** 
Choose the penalty $\lambda$ by $k$-fold cross-validation as in
Section *Cross-validation*, using the cross entropy and then the AUC as
the selection criterion.  Do the two agree?  Explain any difference using the
notebox of Section *Measuring the quality of a classifier*.

**Part e: assessment.** 
Report the confusion matrix, accuracy, precision, recall, $F_1$ and AUC on a
test set touched only once.  Compare against the majority-class baseline.
Then choose a threshold appropriate to a screening application, justify it,
and report how the confusion matrix changes.

**Part f: multiclass.** 
Extend to the softmax (5.27) and apply it to a multiclass data
set -- the handwritten digits included with `scikit-learn` are a
natural choice.  Report the full $K\times K$ confusion matrix and identify
which classes are confused with which.  Keep this result: the next chapter
applies a neural network to the same data, and the comparison is the point.
